# UNet++ for R-Peak Detection on MIT-BIH Arrhythmia Database
This notebook implements a UNet++ architecture specifically designed to detect R-peaks in long-term ECG signals from the MIT-BIH Arrhythmia Database.

**Key Features:**
1. **Inter-Patient Split**: Uses the research gold-standard DS1 (Train) and DS2 (Test) split introduced by De Chazal et al. (2004) to ensure the model generalizes to completely unseen patients.
2. **Mask Generation**: Widens single-point R-peak annotations into a $\pm 10$ sample region ($55\text{ms}$) to create a binary segmentation mask.
3. **Chunking**: Chops continuous 30-minute signals into non-overlapping 2048-sample segments.
4. **UNet++ Architecture**: Employs dense skip connections for precise boundary localization.
5. **Rigorous Evaluation**: Tests the model iteratively over the entire test set (DS2) and outputs standard metrics (Sensitivity, Positive Predictivity, Error Rate) directly comparable to DSP methods.



In [1]:
import wfdb
import numpy as np
import os
import glob
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
import matplotlib.pyplot as plt
import pandas as pd
import time
import scipy.signal as signal

# Set random seed for reproducibility
torch.manual_seed(42)
np.random.seed(42)
plt.rcParams['figure.figsize'] = (12, 4)

# GPU Setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")



OSError: [WinError 1114] A dynamic link library (DLL) initialization routine failed. Error loading "c:\Python 3.11.5\Lib\site-packages\torch\lib\c10.dll" or one of its dependencies.

## 1. Data Processing and Mask Generation


In [ ]:
# Standard MIT-BIH Inter-Patient Split (DS1 for train, DS2 for test)
# DS1: 22 records
DS1_TRAIN = ['101', '106', '108', '109', '112', '114', '115', '116', '118', '119', '122', '124', 
             '201', '203', '205', '207', '208', '209', '215', '220', '223', '230']
# DS2: 22 records (Note: records 102, 104, 107, 217 etc are often excluded or kept in test, we use all available non-DS1 as test)
DS2_TEST = ['100', '103', '105', '111', '113', '117', '121', '123', '200', '202', '210', '212', 
            '213', '214', '219', '221', '222', '228', '231', '232', '233', '234']

CHUNK_SIZE = 2048
PEAK_RADIUS = 10 # +/- 10 samples around the peak marked as class 1

def get_annotated_peaks(record_path):
    ann = wfdb.rdann(record_path, 'atr')
    beat_types = ['N', 'L', 'R', 'B', 'A', 'a', 'J', 'S', 'V', 'r', 'F', 'e', 'j', 'n', 'E', '/', 'f', 'Q', '?']
    beats = [ann.sample[i] for i in range(len(ann.symbol)) if ann.symbol[i] in beat_types]
    return np.array(beats)

def prepare_dataset(records_list):
    X_data = []
    y_data = []
    
    for rec in records_list:
        if not os.path.exists(rec + '.dat'):
            continue
            
        record = wfdb.rdsamp(rec)
        ecg = record[0][:, 0] # channel 0
        peaks = get_annotated_peaks(rec)
        
        # Create full length mask
        mask = np.zeros(len(ecg), dtype=np.int64)
        for p in peaks:
            start = max(0, p - PEAK_RADIUS)
            end = min(len(ecg), p + PEAK_RADIUS + 1)
            mask[start:end] = 1
            
        # Chunking
        num_chunks = len(ecg) // CHUNK_SIZE
        for i in range(num_chunks):
            start = i * CHUNK_SIZE
            end = start + CHUNK_SIZE
            
            chunk_ecg = ecg[start:end]
            chunk_mask = mask[start:end]
            
            # Normalize chunk
            std = np.std(chunk_ecg)
            if std > 0:
                chunk_ecg = (chunk_ecg - np.mean(chunk_ecg)) / std
            else:
                chunk_ecg = chunk_ecg - np.mean(chunk_ecg)
                
            X_data.append(chunk_ecg)
            y_data.append(chunk_mask)
            
    X_tensor = torch.tensor(np.array(X_data), dtype=torch.float32).unsqueeze(1)
    y_tensor = torch.tensor(np.array(y_data), dtype=torch.long)
    return X_tensor, y_tensor

print("Preparing Training Set (DS1)...")
X_train, y_train = prepare_dataset(DS1_TRAIN)
print("Preparing Test Set (DS2)...")
X_test, y_test = prepare_dataset(DS2_TEST)

print(f"Train Shape: X={X_train.shape}, y={y_train.shape}")
print(f"Test Shape:  X={X_test.shape}, y={y_test.shape}")



## 2. UNet++ Architecture


In [ ]:
class ConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv1 = nn.Conv1d(in_channels, out_channels, kernel_size=15, padding=7)
        self.bn1 = nn.BatchNorm1d(out_channels)
        self.relu1 = nn.ReLU()

        self.conv2 = nn.Conv1d(out_channels, out_channels, kernel_size=15, padding=7)
        self.bn2 = nn.BatchNorm1d(out_channels)
        self.relu2 = nn.ReLU()

    def forward(self, x):
        x = self.relu1(self.bn1(self.conv1(x)))
        x = self.relu2(self.bn2(self.conv2(x)))
        return x

class ECGUNetPlusPlus(nn.Module):
    def __init__(self, in_channels=1, num_classes=2):
        super().__init__()
        filters = [16, 32, 64, 128, 256]

        # Encoder (Downsampling path / Column 0)
        self.conv0_0 = ConvBlock(in_channels, filters[0])
        self.pool1 = nn.MaxPool1d(kernel_size=2)
        self.conv1_0 = ConvBlock(filters[0], filters[1])
        self.pool2 = nn.MaxPool1d(kernel_size=2)
        self.conv2_0 = ConvBlock(filters[1], filters[2])
        self.pool3 = nn.MaxPool1d(kernel_size=2)
        self.conv3_0 = ConvBlock(filters[2], filters[3])
        self.pool4 = nn.MaxPool1d(kernel_size=2)

        # Bottleneck
        self.conv4_0 = ConvBlock(filters[3], filters[4])

        # Upsampling Paths and Dense Skip Connections
        # Column 1
        self.up1_0 = nn.ConvTranspose1d(filters[1], filters[0], kernel_size=8, stride=2, padding=3)
        self.conv0_1 = ConvBlock(filters[0]*2, filters[0])
        self.up2_0 = nn.ConvTranspose1d(filters[2], filters[1], kernel_size=8, stride=2, padding=3)
        self.conv1_1 = ConvBlock(filters[1]*2, filters[1])
        self.up3_0 = nn.ConvTranspose1d(filters[3], filters[2], kernel_size=8, stride=2, padding=3)
        self.conv2_1 = ConvBlock(filters[2]*2, filters[2])
        self.up4_0 = nn.ConvTranspose1d(filters[4], filters[3], kernel_size=8, stride=2, padding=3)
        self.conv3_1 = ConvBlock(filters[3]*2, filters[3])

        # Column 2
        self.up1_1 = nn.ConvTranspose1d(filters[1], filters[0], kernel_size=8, stride=2, padding=3)
        self.conv0_2 = ConvBlock(filters[0]*3, filters[0])
        self.up2_1 = nn.ConvTranspose1d(filters[2], filters[1], kernel_size=8, stride=2, padding=3)
        self.conv1_2 = ConvBlock(filters[1]*3, filters[1])
        self.up3_1 = nn.ConvTranspose1d(filters[3], filters[2], kernel_size=8, stride=2, padding=3)
        self.conv2_2 = ConvBlock(filters[2]*3, filters[2])

        # Column 3
        self.up1_2 = nn.ConvTranspose1d(filters[1], filters[0], kernel_size=8, stride=2, padding=3)
        self.conv0_3 = ConvBlock(filters[0]*4, filters[0])
        self.up2_2 = nn.ConvTranspose1d(filters[2], filters[1], kernel_size=8, stride=2, padding=3)
        self.conv1_3 = ConvBlock(filters[1]*4, filters[1])

        # Column 4 (Final Decoder output)
        self.up1_3 = nn.ConvTranspose1d(filters[1], filters[0], kernel_size=8, stride=2, padding=3)
        self.conv0_4 = ConvBlock(filters[0]*5, filters[0])

        # Final output layer
        self.out_conv = nn.Conv1d(filters[0], num_classes, kernel_size=1)

    def pad_to_match(self, x, skip_connection):
        diff = skip_connection.size(-1) - x.size(-1)
        if diff > 0:
            x = F.pad(x, (0, diff))
        return x

    def forward(self, x):
        # Column 0 (Encoder)
        x0_0 = self.conv0_0(x)
        x1_0 = self.conv1_0(self.pool1(x0_0))
        x2_0 = self.conv2_0(self.pool2(x1_0))
        x3_0 = self.conv3_0(self.pool3(x2_0))
        x4_0 = self.conv4_0(self.pool4(x3_0))

        # Column 1
        u1_0 = self.pad_to_match(self.up1_0(x1_0), x0_0)
        x0_1 = self.conv0_1(torch.cat([x0_0, u1_0], dim=1))
        u2_0 = self.pad_to_match(self.up2_0(x2_0), x1_0)
        x1_1 = self.conv1_1(torch.cat([x1_0, u2_0], dim=1))
        u3_0 = self.pad_to_match(self.up3_0(x3_0), x2_0)
        x2_1 = self.conv2_1(torch.cat([x2_0, u3_0], dim=1))
        u4_0 = self.pad_to_match(self.up4_0(x4_0), x3_0)
        x3_1 = self.conv3_1(torch.cat([x3_0, u4_0], dim=1))

        # Column 2
        u1_1 = self.pad_to_match(self.up1_1(x1_1), x0_1)
        x0_2 = self.conv0_2(torch.cat([x0_0, x0_1, u1_1], dim=1))
        u2_1 = self.pad_to_match(self.up2_1(x2_1), x1_1)
        x1_2 = self.conv1_2(torch.cat([x1_0, x1_1, u2_1], dim=1))
        u3_1 = self.pad_to_match(self.up3_1(x3_1), x2_1)
        x2_2 = self.conv2_2(torch.cat([x2_0, x2_1, u3_1], dim=1))

        # Column 3
        u1_2 = self.pad_to_match(self.up1_2(x1_2), x0_2)
        x0_3 = self.conv0_3(torch.cat([x0_0, x0_1, x0_2, u1_2], dim=1))
        u2_2 = self.pad_to_match(self.up2_2(x2_2), x1_2)
        x1_3 = self.conv1_3(torch.cat([x1_0, x1_1, x1_2, u2_2], dim=1))

        # Column 4
        u1_3 = self.pad_to_match(self.up1_3(x1_3), x0_3)
        x0_4 = self.conv0_4(torch.cat([x0_0, x0_1, x0_2, x0_3, u1_3], dim=1))

        out = self.out_conv(x0_4)
        return out



## 3. Loss Functions and Training


In [ ]:
class DiceLoss(nn.Module):
    def __init__(self, smooth=1e-6):
        super().__init__()
        self.smooth = smooth

    def forward(self, logits, targets):
        num_classes = logits.shape[1]
        probs = F.softmax(logits, dim=1)
        targets_onehot = F.one_hot(targets, num_classes=num_classes).permute(0, 2, 1).float()
        intersection = (probs * targets_onehot).sum(dim=(0, 2))
        union = probs.sum(dim=(0, 2)) + targets_onehot.sum(dim=(0, 2))
        dice = (2 * intersection + self.smooth) / (union + self.smooth)
        return 1 - dice.mean()

# Calculate class weights for heavily imbalanced data
all_labels = y_train.flatten()
class_counts = torch.bincount(all_labels)
weights = 1.0 / class_counts.float()
weights = weights / weights.sum() * len(weights)
print(f"Class Weights: {weights}")



In [ ]:
# Create DataLoaders
batch_size = 32
train_dataset = TensorDataset(X_train, y_train)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

# For validation, we use a random 10% of the test set just to guide early stopping
val_size = int(0.1 * len(X_test))
test_size = len(X_test) - val_size
val_dataset, _ = torch.utils.data.random_split(TensorDataset(X_test, y_test), [val_size, test_size])
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

model = ECGUNetPlusPlus(in_channels=1, num_classes=2).to(device)
ce_loss = nn.CrossEntropyLoss(weight=weights.to(device))
dice_loss = DiceLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)

num_epochs = 15
patience = 5
best_val_loss = float('inf')
counter = 0

print("Starting training...")
start_time = time.time()

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    
    for inputs, targets in train_loader:
        inputs, targets = inputs.to(device), targets.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        
        loss_ce = ce_loss(outputs, targets)
        loss_d = dice_loss(outputs, targets)
        loss = 0.5 * loss_ce + 0.5 * loss_d
        
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
        
    avg_train_loss = running_loss / len(train_loader)
    
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for inputs, targets in val_loader:
            inputs, targets = inputs.to(device), targets.to(device)
            outputs = model(inputs)
            loss_ce = ce_loss(outputs, targets)
            loss_d = dice_loss(outputs, targets)
            loss = 0.5 * loss_ce + 0.5 * loss_d
            val_loss += loss.item()
            
    avg_val_loss = val_loss / len(val_loader)
    scheduler.step(avg_val_loss)
    
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        counter = 0
        torch.save(model.state_dict(), "best_unetpp_mitbih.pth")
    else:
        counter += 1
        
    print(f"Epoch [{epoch+1}/{num_epochs}] Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")
    
    if counter >= patience:
        print("Early stopping triggered.")
        break
        
print(f"Training finished in {(time.time() - start_time)/60:.2f} minutes!")
model.load_state_dict(torch.load("best_unetpp_mitbih.pth"))



## 4. Evaluate on Test Set (DS2)
We will now load each record from the test set, chunk it, run inference, extract exact R-peak timestamps from the predicted segmentation masks, and evaluate accuracy.



In [ ]:
def extract_peaks_from_mask(mask, offset=0):
    """
    Converts binary predicted mask blocks (e.g. [0, 1, 1, 1, 0]) into single peak timestamps
    by finding the center of each block.
    """
    peaks = []
    in_peak = False
    start_idx = 0
    for i in range(len(mask)):
        if mask[i] == 1 and not in_peak:
            in_peak = True
            start_idx = i
        elif mask[i] == 0 and in_peak:
            in_peak = False
            center = (start_idx + i) // 2
            peaks.append(center + offset)
    if in_peak:
        center = (start_idx + len(mask)) // 2
        peaks.append(center + offset)
    return peaks

def evaluate_peaks(detected_peaks, annotated_peaks, tolerance_samples=54): # 150ms at 360Hz = 54 samples
    tp = 0; fn = 0
    detected_matched = set()
    for ann in annotated_peaks:
        close_peaks = [p for p in detected_peaks if abs(p - ann) <= tolerance_samples]
        if close_peaks:
            closest = min(close_peaks, key=lambda x: abs(x - ann))
            if closest not in detected_matched:
                tp += 1
                detected_matched.add(closest)
            else:
                fn += 1
        else:
            fn += 1
    fp = len(detected_peaks) - len(detected_matched)
    return tp, fp, fn

results = []
model.eval()

print("Evaluating on DS2 Test Records...")
start_eval_time = time.time()

with torch.no_grad():
    for rec in DS2_TEST:
        if not os.path.exists(rec + '.dat'):
            continue
            
        record = wfdb.rdsamp(rec)
        ecg = record[0][:, 0]
        fs = record[1]['fs']
        ann_peaks = get_annotated_peaks(rec)
        
        det_peaks = []
        num_chunks = len(ecg) // CHUNK_SIZE
        
        t0 = time.time()
        for i in range(num_chunks):
            start = i * CHUNK_SIZE
            chunk_ecg = ecg[start:start+CHUNK_SIZE]
            
            std = np.std(chunk_ecg)
            if std > 0:
                chunk_ecg = (chunk_ecg - np.mean(chunk_ecg)) / std
            else:
                chunk_ecg = chunk_ecg - np.mean(chunk_ecg)
                
            chunk_tensor = torch.tensor(chunk_ecg, dtype=torch.float32).unsqueeze(0).unsqueeze(0).to(device)
            output = model(chunk_tensor)
            pred_mask = torch.argmax(output, dim=1).squeeze(0).cpu().numpy()
            
            chunk_peaks = extract_peaks_from_mask(pred_mask, offset=start)
            det_peaks.extend(chunk_peaks)
            
        proc_time = time.time() - t0
        
        # We only evaluate up to the number of chunks we processed to be fair
        max_idx = num_chunks * CHUNK_SIZE
        valid_ann_peaks = [p for p in ann_peaks if p < max_idx]
        
        tp, fp, fn = evaluate_peaks(det_peaks, valid_ann_peaks)
        results.append({'Record': rec, 'Total Beats': len(valid_ann_peaks), 'TP': tp, 'FP': fp, 'FN': fn, 'Time (s)': proc_time})

print(f"Evaluation complete in {time.time()-start_eval_time:.2f} seconds.")



## 5. Summary Metrics


In [ ]:
df = pd.DataFrame(results)

df['Sensitivity (%)'] = (df['TP'] / (df['TP'] + df['FN']) * 100).fillna(0)
df['Predictivity (%)'] = (df['TP'] / (df['TP'] + df['FP']) * 100).fillna(0)
df['Error Rate (%)'] = ((df['FP'] + df['FN']) / df['Total Beats'] * 100).fillna(0)

total_beats = df['Total Beats'].sum()
total_tp = df['TP'].sum()
total_fp = df['FP'].sum()
total_fn = df['FN'].sum()

overall_se = total_tp / (total_tp + total_fn) * 100
overall_pp = total_tp / (total_tp + total_fp) * 100
overall_er = (total_fp + total_fn) / total_beats * 100

print("="*50)
print("OVERALL PERFORMANCE (TEST SET / DS2)")
print("="*50)
print(f"Total Beats Evaluated : {total_beats}")
print(f"True Positives (TP)   : {total_tp}")
print(f"False Positives (FP)  : {total_fp}")
print(f"False Negatives (FN)  : {total_fn}")
print(f"Sensitivity (Se)      : {overall_se:.2f} %")
print(f"Positive Predictivity : {overall_pp:.2f} %")
print(f"Detection Error Rate  : {overall_er:.2f} %")
print("="*50)

df.to_csv('unetpp_mitbih_results.csv', index=False)
print("\nResults saved to 'unetpp_mitbih_results.csv'")
display(df.round(2))

